# blob_decoder
Pure OCF-LZW decode cascade + OCF/magic helpers + shared constants, factored out of
`blob_shared_lib` so both the text pipeline (Blob 1/2/3 via blob_shared_lib %run) and
`Blob 4:Files` share ONE decoder. Pure functions + constants only: no %pip, no preflight,
no spark/dbutils — loads on serverless. The `ocflzw` import is guarded.

Sliced byte-verbatim from prod blob_shared_lib cells 2-6 by CC/_blob_decoder_split_gen;
classify_file_type appended for Blob 4:Files.

In [0]:
import hashlib, io, json, os, re, time
from typing import Optional, Tuple

# ==================== VERSION CONSTANTS ====================
# Bump exactly one of these when the corresponding stage changes.
# Bumping invalidates all rows with a lower version; the replay notebook
# will re-run the affected stage when its scope predicate matches.

DECOMPRESSOR_VERSION = 2
PARSER_VERSION = 2
POST_PROCESSOR_VERSION = 3

# ==================== SIZE LIMITS ====================
MAX_BLOB_SIZE = 16 * 1024 * 1024  # 16 MB; rows above this are flagged "Compressed Too Large"
OCR_MAX_PAGES = 10
OCR_MAX_PDF_SIZE_MB = 50
OCR_LANG = "eng"
# pytesseract's `timeout=` is forwarded to the tesseract subprocess and actually
# kills it (not a Python-side abort). Prevents pathological PDFs from hanging.
OCR_PAGE_TIMEOUT_SEC = 30

# PDF hang defenses — see ParsePool below.
# Size cap skips decompression-DoS-scale inputs before we ever touch pypdfium2 /
# PyMuPDF; the subprocess timeout handles small-but-pathological PDFs that hang
# native code paths which have no Python-side timeout hook.
PDF_EXTRACT_MAX_BYTES = 20 * 1024 * 1024  # 20 MB — skip parsing, mark as "[PDF - too large]"
PARSE_TIMEOUT_SEC = 30  # hard wall-clock cap per parse() call, enforced via subprocess

# ==================== BYTE MARKERS ====================
OCF_MARKER = b'ocf_blob\0'
PDF_MAGIC = b'%PDF-'
RTF_MAGIC = b'{\\rtf'
RTF_MAGIC_LOOSE = b'{\\'
ZIP_MAGIC = b'\x50\x4B\x03\x04'
OLE_MAGIC = b'\xD0\xCF\x11\xE0\xA1\xB1\x1A\xE1'
TIFF_LE_MAGIC = b'II*\x00'
TIFF_BE_MAGIC = b'MM\x00*'

CONTAINER_MAGICS = [PDF_MAGIC, RTF_MAGIC, ZIP_MAGIC, OLE_MAGIC, TIFF_LE_MAGIC, TIFF_BE_MAGIC]

# ==================== BINARY CONTENT TYPES ====================
BINARY_CONTENT_TYPES = frozenset({
    'image/tiff', 'image/png', 'image/g3fax', 'image/x-tga', 'image/jpeg',
    'application/x-dosexec', 'application/x-matlab-data', 'application/x-coff',
    'audio/x-mp4a-latm', 'audio/mpeg',
    'application/zlib', 'application/x-lzma', 'application/x-compress',
    'application/SIMH-tape-data', 'font/x-amiga-font',
    'application/x-executable', 'application/x-sharedlib',
})

# ==================== SANITY LIMITS ====================
BLOB_LENGTH_TOLERANCE = 0.01  # decompressed length within 1% of BLOB_LENGTH before flagging truncation
LZW_LOOP_BUDGET_MULT = 16     # max LZW iterations = compressed_size * this

# ==================== CONTENT HASHING ====================

def raw_content_sha256(chunks: list) -> str:
    """Stable hash of concatenated raw chunk bytes (in BLOB_SEQ_NUM order).

    Input: list of (BLOB_SEQ_NUM, BLOB_CONTENTS) tuples or dict-like rows.
    Output: 64-char hex digest.
    """
    h = hashlib.sha256()
    for c in sorted(chunks, key=lambda x: x['BLOB_SEQ_NUM'] or 0):
        contents = c['BLOB_CONTENTS']
        if contents:
            h.update(contents)
    return h.hexdigest()


In [0]:

# ==================== CERNER LZW DECODER (pgodwin-port) ====================
# Python port of pgodwin/OcfLzw2/LZWDecoder.cs with three hardening changes:
#   1. Growable bytearray output (replaces fixed 8192 buffer)
#   2. Output bytes masked with & 0xFF
#   3. Loop-iteration budget instead of signal-based timeout

class CernerLZWError(Exception):
    """Raised by cerner_lzw_decompress on unrecoverable input."""


def cerner_lzw_decompress(data: bytes, *, loop_budget: Optional[int] = None, max_output_bytes: int = 256 * 1024 * 1024) -> bytes:
    """Decompress Cerner OCF LZW bytes.

    Format: MSB-first bit packing; 9 -> 13 bit variable code width; clear=256;
    EOF=257; code width bumps at code boundaries 511, 1023, 2047, 4095
    (early-change: width increases one code before the dictionary fills that
    width, matching the Cerner encoder).

    Args:
        data: Raw compressed bytes with OCF wrapper already stripped.
        loop_budget: Max iterations; defaults to len(data) * LZW_LOOP_BUDGET_MULT.
        max_output_bytes: Max output size; defaults to 256 MiB.

    Returns:
        Decompressed bytes.

    Raises:
        CernerLZWError: on malformed input that cannot be recovered.
    """
    if not data:
        return b""

    CLEAR_CODE = 256
    EOI_CODE = 257
    FIRST_CODE = 258
    MAX_CODE = (1 << 13) - 1  # 8191; pgodwin uses 13-bit max dictionary

    dictionary = {i: bytes([i & 0xFF]) for i in range(256)}
    next_code = FIRST_CODE
    code_size = 9

    output = bytearray()

    def _check_output_size():
        if len(output) > max_output_bytes:
            raise CernerLZWError(f"output size {len(output)} exceeds max {max_output_bytes}")

    prev_string = b""

    # MSB-first bit buffer
    bit_buffer = 0
    bits_in_buffer = 0
    byte_pos = 0
    data_len = len(data)

    def read_code():
        nonlocal bit_buffer, bits_in_buffer, byte_pos
        while bits_in_buffer < code_size and byte_pos < data_len:
            bit_buffer = (bit_buffer << 8) | data[byte_pos]
            bits_in_buffer += 8
            byte_pos += 1
        if bits_in_buffer < code_size:
            return None
        code = (bit_buffer >> (bits_in_buffer - code_size)) & ((1 << code_size) - 1)
        bits_in_buffer -= code_size
        bit_buffer &= (1 << bits_in_buffer) - 1
        return code

    if loop_budget is None:
        loop_budget = max(1024, data_len * LZW_LOOP_BUDGET_MULT)

    iterations = 0

    # Read first code (allowing a leading clear)
    code = read_code()
    if code is None:
        return bytes(output)
    if code == CLEAR_CODE:
        code = read_code()
        if code is None or code == EOI_CODE:
            return bytes(output)

    if code < 256:
        output.append(code & 0xFF)
        _check_output_size()
        prev_string = bytes([code & 0xFF])
    elif code in dictionary:
        output.extend(dictionary[code])
        _check_output_size()
        prev_string = dictionary[code]
    else:
        raise CernerLZWError(f"first code {code} is not a literal or dictionary entry")

    while iterations < loop_budget:
        iterations += 1
        code = read_code()
        if code is None or code == EOI_CODE:
            break
        if code == CLEAR_CODE:
            dictionary = {i: bytes([i & 0xFF]) for i in range(256)}
            next_code = FIRST_CODE
            code_size = 9
            prev_string = b""
            code = read_code()
            if code is None or code == EOI_CODE:
                break
            if code < 256:
                output.append(code & 0xFF)
                _check_output_size()
                prev_string = bytes([code & 0xFF])
            elif code in dictionary:
                output.extend(dictionary[code])
                _check_output_size()
                prev_string = dictionary[code]
            else:
                raise CernerLZWError(f"post-clear code {code} invalid")
            continue

        if code < 256:
            current_string = bytes([code & 0xFF])
        elif code in dictionary:
            current_string = dictionary[code]
        elif code == next_code and prev_string:
            # KwKwK
            current_string = prev_string + prev_string[0:1]
        else:
            raise CernerLZWError(f"unknown code {code} at iter {iterations} "
                                 f"(next_code={next_code}, dict_size={len(dictionary)})")

        output.extend(current_string)
        _check_output_size()

        if prev_string and next_code <= MAX_CODE:
            dictionary[next_code] = prev_string + current_string[0:1]
            next_code += 1
            # Early-change width bumps: width increases one code before the
            # dictionary fills that width (matches the Cerner encoder output
            # and the working TIFF-variant fallback below). The previous
            # late-change thresholds (512/1024/2048/4096) were off-by-one
            # and caused bit-stream desync at each width boundary.
            if next_code == 511 and code_size == 9:
                code_size = 10
            elif next_code == 1023 and code_size == 10:
                code_size = 11
            elif next_code == 2047 and code_size == 11:
                code_size = 12
            elif next_code == 4095 and code_size == 12:
                code_size = 13
        prev_string = current_string
    else:
        raise CernerLZWError(f"loop budget {loop_budget} exceeded")

    return bytes(output)


In [0]:

# ==================== OCF WRAPPER + MAGIC SNIFF ====================

def strip_ocf_aggressive(data: bytes) -> bytes:
    if not data:
        return data
    if data.endswith(OCF_MARKER):
        data = data[:-len(OCF_MARKER)]
    if OCF_MARKER in data:
        data = b"".join(data.split(OCF_MARKER))
    return data


def strip_ocf_conservative(data: bytes) -> bytes:
    if not data:
        return data
    if data.endswith(OCF_MARKER):
        return data[:-len(OCF_MARKER)]
    return data


def sniff_container_magic(data: bytes) -> bool:
    """True if data begins with a known container magic (PDF/RTF/ZIP/OLE/TIFF)."""
    if not data:
        return False
    head = data[:16]
    if head.startswith(RTF_MAGIC):
        return True
    for magic in (PDF_MAGIC, ZIP_MAGIC, OLE_MAGIC, TIFF_LE_MAGIC, TIFF_BE_MAGIC):
        if head.startswith(magic):
            return True
    # loose RTF: '{\\' — only trust if followed by an ASCII letter
    if head.startswith(RTF_MAGIC_LOOSE) and len(head) > 2 and 0x20 <= head[2] < 0x7F:
        return True
    return False


In [0]:

# ==================== FALLBACK LZW DECODERS (ported from v1) ====================

def tolerant_lzw_decompress(data: bytes) -> bytes:
    """LSB-first tolerant decoder — fallback for blobs the pgodwin port can't handle.
    Carried over from blob_processing v1 with minor cleanup.
    """
    if not data:
        return b""
    CLEAR_CODE, END_CODE, FIRST_CODE, MAX_CODE = 256, 257, 258, 65535
    dictionary = {i: bytes([i]) for i in range(256)}
    next_code = FIRST_CODE
    output = bytearray()
    prev_string = b""
    code_size = 9
    max_code_for_size = 512
    bit_buffer = 0
    bits_in_buffer = 0
    byte_pos = 0
    data_len = len(data)
    max_iters = data_len * 8
    iters = 0
    while iters < max_iters:
        while bits_in_buffer < code_size and byte_pos < data_len:
            bit_buffer |= data[byte_pos] << bits_in_buffer
            bits_in_buffer += 8
            byte_pos += 1
        if bits_in_buffer < code_size:
            break
        code = bit_buffer & ((1 << code_size) - 1)
        bit_buffer >>= code_size
        bits_in_buffer -= code_size
        iters += 1
        if code == END_CODE:
            break
        if code == CLEAR_CODE:
            dictionary = {i: bytes([i]) for i in range(256)}
            next_code = FIRST_CODE
            code_size = 9
            max_code_for_size = 512
            prev_string = b""
            continue
        if code < 256:
            current_string = bytes([code])
        elif code in dictionary:
            current_string = dictionary[code]
        elif code == next_code and prev_string:
            current_string = prev_string + prev_string[0:1]
        else:
            if prev_string:
                current_string = prev_string + prev_string[0:1]
            else:
                continue
        output.extend(current_string)
        if prev_string and next_code <= MAX_CODE:
            dictionary[next_code] = prev_string + current_string[0:1]
            next_code += 1
            if next_code >= max_code_for_size and code_size < 16:
                code_size += 1
                max_code_for_size *= 2
        prev_string = current_string
    return bytes(output)


def tiff_lzw_decompress(data: bytes) -> bytes:
    """TIFF-variant LZW (MSB-first with early-change quirk). 8192-entry max.
    Carried over from blob_processing v1 as a third-tier fallback.
    """
    if not data:
        return b""
    CLEAR_CODE, EOI_CODE, FIRST_CODE, MAX_DICT_SIZE = 256, 257, 258, 8192
    dictionary = {i: bytes([i]) for i in range(256)}
    next_code = FIRST_CODE
    code_size = 9
    output = bytearray()
    prev_string = b""
    bit_buffer = 0
    bits_in_buffer = 0
    byte_pos = 0
    data_len = len(data)

    def read_code():
        nonlocal bit_buffer, bits_in_buffer, byte_pos
        while bits_in_buffer < code_size and byte_pos < data_len:
            bit_buffer = (bit_buffer << 8) | data[byte_pos]
            bits_in_buffer += 8
            byte_pos += 1
        if bits_in_buffer < code_size:
            return None
        c = (bit_buffer >> (bits_in_buffer - code_size)) & ((1 << code_size) - 1)
        bits_in_buffer -= code_size
        bit_buffer &= (1 << bits_in_buffer) - 1
        return c

    code = read_code()
    if code is None:
        return bytes(output)
    if code == CLEAR_CODE:
        code = read_code()
        if code is None or code == EOI_CODE:
            return bytes(output)
    if code < 256:
        output.append(code)
        prev_string = bytes([code])
    elif code in dictionary:
        output.extend(dictionary[code])
        prev_string = dictionary[code]
    else:
        return bytes(output)

    max_iters = data_len * 10
    for _ in range(max_iters):
        code = read_code()
        if code is None or code == EOI_CODE:
            break
        if code == CLEAR_CODE:
            dictionary = {i: bytes([i]) for i in range(256)}
            next_code = FIRST_CODE
            code_size = 9
            prev_string = b""
            code = read_code()
            if code is None or code == EOI_CODE:
                break
            if code < 256:
                output.append(code)
                prev_string = bytes([code])
            elif code in dictionary:
                output.extend(dictionary[code])
                prev_string = dictionary[code]
            continue
        if code < 256:
            current_string = bytes([code])
        elif code in dictionary:
            current_string = dictionary[code]
        elif code == next_code and prev_string:
            current_string = prev_string + prev_string[0:1]
        else:
            continue
        output.extend(current_string)
        if prev_string and next_code < MAX_DICT_SIZE:
            dictionary[next_code] = prev_string + current_string[0:1]
            next_code += 1
            if next_code == 511 and code_size == 9:
                code_size = 10
            elif next_code == 1023 and code_size == 10:
                code_size = 11
            elif next_code == 2047 and code_size == 11:
                code_size = 12
            elif next_code == 4095 and code_size == 12:
                code_size = 13
        prev_string = current_string
    return bytes(output)


In [0]:

# ==================== DECOMPRESS DISPATCHER ====================

try:
    from ocflzw_decompress.lzw import LzwDecompress
    _HAVE_OCFLZW = True
except ImportError:
    _HAVE_OCFLZW = False


def decompress(raw: bytes, compression_cd, chunk_count: int, blob_length: Optional[int], metrics: dict):
    """Main decompression entry point.

    Sets these keys in `metrics`:
        decompression_strategy, ocf_marker_count, blob_length_mismatch,
        blob_length_expected, blob_length_actual
    Returns (decompressed_bytes, error_message).
    On success error_message is None. On failure decompressed_bytes is None.
    """
    if not raw:
        return None, "Empty content"

    metrics["ocf_marker_count"] = raw.count(OCF_MARKER)

    try:
        cd = int(compression_cd) if compression_cd is not None else None
    except Exception:
        cd = None

    # Uncompressed path
    if cd == 727:
        metrics["decompression_strategy"] = "uncompressed"
        out = strip_ocf_aggressive(raw) if metrics["ocf_marker_count"] > 0 else raw
        _set_length_metrics(out, blob_length, metrics)
        return out, None

    if cd != 728:
        return None, f"Unknown compression: {compression_cd}"

    # Magic-byte sniff after stripping OCF — if the content is already a known
    # container, don't run LZW on it.
    stripped = strip_ocf_aggressive(raw)
    if sniff_container_magic(stripped):
        metrics["decompression_strategy"] = "uncompressed_sniffed"
        _set_length_metrics(stripped, blob_length, metrics)
        return stripped, None

    # LZW cascade: pgodwin port -> ocflzw -> tiff variant -> tolerant variant.
    # Every tier uses `if result:` so that an empty / zero-byte decode is treated
    # as a failure and we fall through to the next tier rather than silently
    # reporting an empty success.
    strategies = [("aggressive", strip_ocf_aggressive), ("conservative", strip_ocf_conservative), ("raw", lambda d: d)]
    last_error = None

    # Tier 1: pgodwin port
    for sname, sfn in strategies:
        try:
            cleaned = sfn(raw)
            result = cerner_lzw_decompress(cleaned)
            if result:
                metrics["decompression_strategy"] = f"pgodwin_port_{sname}"
                _set_length_metrics(result, blob_length, metrics)
                return result, None
        except Exception as e:
            last_error = f"pgodwin_{sname}: {e}"

    # Tier 2: ocflzw library
    if _HAVE_OCFLZW:
        for sname, sfn in strategies:
            try:
                cleaned = sfn(raw)
                result = bytes(LzwDecompress().decompress(cleaned))
                if result:
                    metrics["decompression_strategy"] = f"ocflzw_{sname}"
                    _set_length_metrics(result, blob_length, metrics)
                    return result, None
            except Exception as e:
                last_error = f"ocflzw_{sname}: {e}"

    # Tier 3: TIFF variant
    try:
        result = tiff_lzw_decompress(strip_ocf_aggressive(raw))
        if result:
            metrics["decompression_strategy"] = "tiff_fallback"
            _set_length_metrics(result, blob_length, metrics)
            return result, None
    except Exception as e:
        last_error = f"tiff: {e}"

    # Tier 4: tolerant LSB variant
    try:
        result = tolerant_lzw_decompress(strip_ocf_aggressive(raw))
        if result:
            metrics["decompression_strategy"] = "tolerant_fallback"
            _set_length_metrics(result, blob_length, metrics)
            return result, None
    except Exception as e:
        last_error = f"tolerant: {e}"

    return None, f"LZW failed all tiers: {last_error}"


def _set_length_metrics(decompressed: bytes, blob_length: Optional[int], metrics: dict) -> None:
    if decompressed is None:
        return
    metrics["blob_length_actual"] = len(decompressed)
    metrics["blob_length_expected"] = blob_length
    if blob_length and blob_length > 0:
        diff = abs(len(decompressed) - blob_length) / float(blob_length)
        metrics["blob_length_mismatch"] = bool(diff > BLOB_LENGTH_TOLERANCE)
    else:
        metrics["blob_length_mismatch"] = False


In [0]:
# ==================== FILE-TYPE CLASSIFIER (Blob 4:Files) ====================
# Map decoded bytes -> (file_ext, subfolder, content_type) for in-scope inline file types.
# Returns None for anything else (RTF/HTML/DOCX/plaintext/DICOM-pointer/etc).
JPEG_MAGIC = b'\xFF\xD8\xFF'
PNG_MAGIC = b'\x89PNG\r\n\x1a\n'

def classify_file_type(data: bytes):
    """-> (file_ext, subfolder, content_type) or None if not an in-scope file type."""
    if not data:
        return None
    h = data[:16]
    if h.startswith(PDF_MAGIC):
        return ('pdf', 'pdf', 'application/pdf')
    if h.startswith(JPEG_MAGIC):
        return ('jpg', 'jpeg', 'image/jpeg')
    if h.startswith(PNG_MAGIC):
        return ('png', 'png', 'image/png')
    if h.startswith(TIFF_LE_MAGIC) or h.startswith(TIFF_BE_MAGIC):
        return ('tif', 'tiff', 'image/tiff')
    return None

In [0]:
print(f"blob_decoder loaded: decompressor={DECOMPRESSOR_VERSION} have_ocflzw={_HAVE_OCFLZW}")